# 객체 탐지 모델의 성능 테스트

In [11]:
!pip install ultralytics
import ultralytics
!pip install tflite-runtime
ultralytics.checks()

Ultralytics 8.3.243 🚀 Python-3.10.14 torch-2.9.1 CPU (Apple M3 Max)
Setup complete ✅ (14 CPUs, 36.0 GB RAM, 441.3/926.4 GB disk)


In [2]:
# best.pt와 같은 폴더에 두고 실행
from ultralytics import YOLO
from IPython.display import Image, display
import os

model = YOLO('/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/v3.pt')  # 로컬 파일 경로

# 테스트 이미지 직접 지정
img_path = '/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/test3.png'  # 로컬 사진 경로
results = model(img_path, save=True, conf=0.4)


image 1/1 /Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/data/test3.png: 640x640 4 pills, 50.8ms
Speed: 1.9ms preprocess, 50.8ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/src/runs/detect/predict9


In [8]:
from ultralytics import YOLO

model = YOLO('/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/v3_int8.tflite', task='detect')

model.info()

TypeError: model='/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/v3_int8.tflite' should be a *.pt PyTorch model to run this method, but is a different format. PyTorch models can train, val, predict and export, i.e. 'model.train(data=...)', but exported formats like ONNX, TensorRT etc. only support 'predict' and 'val' modes, i.e. 'yolo predict model=yolo11n.onnx'.
To run CUDA or MPS inference please pass the device argument directly in your inference command, i.e. 'model.predict(source=..., device=0)'

In [12]:
import cv2
from ultralytics import YOLO

model = YOLO('/Users/hsj/Documents/GitHub/alyak-deep-learning-app/deep_learning_prj/v3_int8.tflite') 

# MacBook: 1번 카메라 (0번 실패시)
cap = cv2.VideoCapture(1)  # ← 1로 변경!

# 해상도 설정 (M3 Max 최적)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)

if not cap.isOpened():
    print("❌ 모든 카메라 실패! USB 웹캠 연결")
    exit()

print("✅ 실시간 알약 탐지 시작! 'q'키 종료")
print("손바닥에 알약 올려보세요!")

while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠️ 프레임 끊김, 재연결...")
        continue
    
    # YOLO 실시간 탐지
    results = model(frame, conf=0.4, verbose=False)
    annotated = results[0].plot()
    
    cv2.imshow('Alyak 실시간 알약 탐지', annotated)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
✅ 실시간 알약 탐지 시작! 'q'키 종료
손바닥에 알약 올려보세요!


AttributeError: module 'tensorflow' has no attribute 'lite'

In [3]:
# 3. 응답 결과(JSON 형태) 파싱해서 보기
for result in results:
    print(f"--- 탐지 결과 ---")
    boxes = result.boxes
    for box in boxes:
        c = box.cls.item()      # 클래스 번호
        name = result.names[c]  # 클래스 이름
        conf = box.conf.item()  # 확신도
        print(f"객체: {name}, 확신도: {conf:.2f}, 좌표: {box.xyxy[0].tolist()}")

--- 탐지 결과 ---
객체: pill, 확신도: 0.84, 좌표: [229.5972442626953, 124.73421478271484, 348.4969482421875, 236.0360107421875]
객체: pill, 확신도: 0.83, 좌표: [142.31190490722656, 71.18375396728516, 261.0561218261719, 213.6181182861328]
객체: pill, 확신도: 0.83, 좌표: [177.355712890625, 241.96261596679688, 299.57421875, 358.83563232421875]
객체: pill, 확신도: 0.82, 좌표: [11.219368934631348, 70.6978988647461, 113.15438079833984, 173.76710510253906]
